# Three early Pump It Up submissions

Generate two different untuned tree models and their soft-voting ensemble. All three models fit on the complete labelled data; train/validation splitting and model evaluation are deliberately deferred to the formal modelling phase.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEED = 20260815
CLASS_LABELS = np.array([
    'functional',
    'functional needs repair',
    'non functional',
])

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != '2026-08-15-early-experiments':
    raise RuntimeError('Run this notebook from its own dated submission directory.')
STAGE_DIR = NOTEBOOK_DIR.parents[1]
DATA_DIR = STAGE_DIR / 'data'
SRC_DIR = STAGE_DIR / 'src'
sys.path.insert(0, str(SRC_DIR))

from data_preparation import remove_known_redundant_columns
from source_data_validation import validate_aligned_ids, validate_label_frame

raw_train = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_test = pd.read_csv(DATA_DIR / 'TestSetValues.csv')
submission_template = pd.read_csv(DATA_DIR / 'SubmissionFormat.csv')

validate_label_frame(labels)
validate_aligned_ids(raw_train, labels)
if not raw_test['id'].equals(submission_template['id']):
    raise ValueError('Test IDs do not match the submission template in row order.')
if set(labels['status_group']) != set(CLASS_LABELS):
    raise ValueError('The training labels differ from the three documented classes.')

y = labels['status_group'].copy()
print(f'Training rows: {len(raw_train):,}; test rows: {len(raw_test):,}')

Training rows: 59,400; test rows: 14,850


## Audit-led feature preparation

This intentionally applies only the fixed structural and placeholder decisions already supported by the data audit.

In [2]:
def prepare_features(raw_features: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    prepared = remove_known_redundant_columns(raw_features)
    identifiers = prepared.pop('id').copy()

    recorded = pd.to_datetime(prepared.pop('date_recorded'), errors='raise')
    prepared['recorded_year'] = recorded.dt.year.astype('float64')
    prepared['recorded_month'] = recorded.dt.month.astype('float64')
    prepared['recorded_dayofyear'] = recorded.dt.dayofyear.astype('float64')

    construction_year = prepared['construction_year'].replace(0, np.nan)
    pump_age = prepared['recorded_year'] - construction_year
    prepared['pump_age'] = pump_age.where(pump_age.between(0, 100))
    prepared['construction_year_missing'] = construction_year.isna().astype('int8')

    invalid_coordinates = prepared['longitude'].eq(0) | prepared['latitude'].eq(0)
    prepared['coordinates_missing'] = invalid_coordinates.astype('int8')
    prepared.loc[invalid_coordinates, ['longitude', 'latitude']] = np.nan
    prepared['gps_height_zero'] = prepared['gps_height'].eq(0).astype('int8')
    prepared['population_zero'] = prepared['population'].eq(0).astype('int8')

    # These are documented location codes, not quantities with meaningful distances.
    prepared['region_code'] = prepared['region_code'].astype('string')
    prepared['district_code'] = prepared['district_code'].astype('string')

    categorical = prepared.select_dtypes(exclude=['number']).columns
    prepared[categorical] = (
        prepared[categorical]
        .astype('string')
        .fillna('__MISSING__')
        .astype(str)
    )
    return identifiers, prepared

train_ids, X = prepare_features(raw_train)
test_ids, X_test = prepare_features(raw_test)
if list(X.columns) != list(X_test.columns):
    raise ValueError('Prepared training and test predictors do not align.')

numeric_columns = X.select_dtypes(include=['number']).columns.tolist()
categorical_columns = X.columns.difference(numeric_columns, sort=False).tolist()
print(f'Prepared predictors: {X.shape[1]} ({len(numeric_columns)} numeric, {len(categorical_columns)} categorical)')

Prepared predictors: 43 (15 numeric, 28 categorical)


In [3]:
def make_preprocessing() -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ])
    categorical_pipeline = Pipeline([
        ('encode', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            encoded_missing_value=-1,
            dtype=np.float32,
        )),
    ])
    return ColumnTransformer([
        ('numeric', numeric_pipeline, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns),
    ])

def make_extra_trees() -> Pipeline:
    model = ExtraTreesClassifier(
        n_estimators=500,
        max_features=0.7,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=SEED,
    )
    return Pipeline([('prepare', make_preprocessing()), ('model', model)])

def make_hist_gradient_boosting() -> Pipeline:
    model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        max_features=0.8,
        early_stopping=False,
        random_state=SEED,
    )
    return Pipeline([('prepare', make_preprocessing()), ('model', model)])

def ordered_probabilities(model: Pipeline, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    probability_frame = pd.DataFrame(probabilities, columns=model.classes_)
    return probability_frame.loc[:, CLASS_LABELS].to_numpy()

## Fit on all labelled rows and write the submissions

In [4]:
models = {
    'extra_trees': make_extra_trees(),
    'hist_gradient_boosting': make_hist_gradient_boosting(),
}

full_test_probabilities = {}
for name, model in models.items():
    started = time.perf_counter()
    model.fit(X, y)
    full_test_probabilities[name] = ordered_probabilities(model, X_test)
    print(f'Fit {name} on all labelled rows and scored test rows in {time.perf_counter()-started:.1f}s')

full_test_probabilities['soft_vote_ensemble'] = 0.5 * (
    full_test_probabilities['extra_trees']
    + full_test_probabilities['hist_gradient_boosting']
)

output_files = {
    'extra_trees': '01-extra-trees.csv',
    'hist_gradient_boosting': '02-hist-gradient-boosting.csv',
    'soft_vote_ensemble': '03-soft-vote-ensemble.csv',
}

submission_summaries = []
for name, filename in output_files.items():
    predictions = CLASS_LABELS[full_test_probabilities[name].argmax(axis=1)]
    submission = submission_template.loc[:, ['id']].copy()
    submission['status_group'] = predictions
    output_path = NOTEBOOK_DIR / filename
    submission.to_csv(output_path, index=False)

    reloaded = pd.read_csv(output_path)
    if list(reloaded.columns) != ['id', 'status_group']:
        raise ValueError(f'{filename} has the wrong columns.')
    if len(reloaded) != len(submission_template):
        raise ValueError(f'{filename} has the wrong row count.')
    if not reloaded['id'].equals(submission_template['id']):
        raise ValueError(f'{filename} has IDs in the wrong order.')
    if reloaded.isna().any().any() or not set(reloaded['status_group']).issubset(CLASS_LABELS):
        raise ValueError(f'{filename} contains an invalid prediction.')

    shares = reloaded['status_group'].value_counts(normalize=True).reindex(CLASS_LABELS, fill_value=0)
    submission_summaries.append({
        'file': filename,
        'rows': len(reloaded),
        **{label: shares[label] for label in CLASS_LABELS},
    })

summary = pd.DataFrame(submission_summaries).set_index('file')
print('Validated generated submissions; prediction shares:')
print(summary.round(4))

Fit extra_trees on all labelled rows and scored test rows in 12.2s


Fit hist_gradient_boosting on all labelled rows and scored test rows in 9.0s


Validated generated submissions; prediction shares:
                                rows  functional  functional needs repair  \
file                                                                        
01-extra-trees.csv             14850      0.5683                   0.0560   
02-hist-gradient-boosting.csv  14850      0.6053                   0.0301   
03-soft-vote-ensemble.csv      14850      0.5896                   0.0432   

                               non functional  
file                                           
01-extra-trees.csv                     0.3757  
02-hist-gradient-boosting.csv          0.3646  
03-soft-vote-ensemble.csv              0.3672  
